In [25]:
import numpy as np # handle numbers
import pandas as pd # handle data

In [26]:
import joblib # saves and loads trained models

In [ ]:
from sklearn.pipeline import Pipeline # automates individual steps from raw data to model deployment reducing human error and manual effort
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score # GridSearchCV tries different parameters to find the best model, cross_val_score checks model performance across multiple splits
from sklearn.preprocessing import StandardScaler # scales data so all features are comparable
from sklearn.linear_model import LogisticRegression # baseline
from sklearn.svm import SVC # strong classifier
from sklearn.ensemble import RandomForestClassifier # ensemble

In [28]:
from sklearn.metrics import (
accuracy_score, precision_score, recall_score,
f1_score, confusion_matrix, classification_report,
roc_auc_score
)

# accuracy = overall correctness
# precision = how many predicted positives are correct
# recall = how many actual positives you found
# f1 score = balance of precision + recall
# confusion matrix = actual vs predicted table
# classification report = all metric together
# roc-auc = quality of probability predictors

In [29]:
train = pd.read_csv("../train_data.csv")
test = pd.read_csv("../test_data.csv")

In [30]:
feature_names = [
    "radius_mean","texture_mean","perimeter_mean","area_mean","smoothness_mean",
    "compactness_mean","concavity_mean","concave points_mean","symmetry_mean","fractal_dimension_mean",
    "radius_se","texture_se","perimeter_se","area_se","smoothness_se",
    "compactness_se","concavity_se","concave points_se","symmetry_se","fractal_dimension_se",
    "radius_worst","texture_worst","perimeter_worst","area_worst","smoothness_worst",
    "compactness_worst","concavity_worst","concave points_worst","symmetry_worst","fractal_dimension_worst"
]

In [31]:
train.columns = feature_names + ["target"]
test.columns = feature_names + ["target"]

In [32]:
X_train = train.drop("target", axis = 1) # input features
y_train = train["target"] # output labels

X_test = test.drop("target", axis = 1)
y_test = test["target"]

In [ ]:
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=10000))
]) # 1. scale data, 2. train logistic regression

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(probability=True))
]) # 

rf_pipeline = Pipeline([
    ("model", RandomForestClassifier())
]) # no scaler needed because trees don't care about feature scale it cares about thresholds

In [ ]:
lr_pipeline.fit(X_train, y_train)
svm_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

# each model learns patterns from training data

print("\nAll models trained successfully")


All models trained successfully


In [ ]:
lr_pred = lr_pipeline.predict(X_test)
svm_pred = svm_pipeline.predict(X_test)
rf_pred = rf_pipeline.predict(X_test)
# predicts output for unseen data

In [ ]:
svm_probs = svm_pipeline.predict_proba(X_test)[:,1] # gets probability of class 1
roc = roc_auc_score(y_test, svm_probs) # measures how well model separates classes (positive vs negative), close to 1 = better (auc score)

In [37]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "SVM", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, svm_pred),
        accuracy_score(y_test, rf_pred)
    ]
})

In [38]:
print("\nModel Comparison:")
print(results)


Model Comparison:
                 Model  Accuracy
0  Logistic Regression  0.964912
1                  SVM  0.973684
2        Random Forest  0.973684


In [39]:
print("\nROC-AUC (SVM):", roc)


ROC-AUC (SVM): 0.9947089947089947


In [40]:
print("\n--- SVM ---")
print("Accuracy:", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(y_test, svm_pred))
print("Recall:", recall_score(y_test, svm_pred))
print("F1 Score:", f1_score(y_test, svm_pred))


--- SVM ---
Accuracy: 0.9736842105263158
Precision: 1.0
Recall: 0.9285714285714286
F1 Score: 0.9629629629629629


In [41]:
print("\nConfusion Matrix (SVM):")
print(confusion_matrix(y_test, svm_pred))


Confusion Matrix (SVM):
[[72  0]
 [ 3 39]]


In [42]:
print("\nClassification Report (SVM):")
print(classification_report(y_test, svm_pred))


Classification Report (SVM):
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        72
           1       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114



In [ ]:
# hyperparameter tuning

param_grid = {
    "model__C": [0.1, 1, 10], # C controls strictness
    "model__kernel": ["linear", "rbf"] # kernel is a type of boundary (shape)
    # find which combination gives the best performance
}

grid = GridSearchCV(svm_pipeline, param_grid, cv=5) #tries all combination using 5-fold cross-validation
grid.fit(X_train, y_train) # best parameters

print("\nBest Parameters:", grid.best_params_) # best settings

best_model = grid.best_estimator_ # best version of model


Best Parameters: {'model__C': 1, 'model__kernel': 'rbf'}


In [ ]:
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5) # splits data into five parts and tests repeatedly
print("Cross Validation Accuracy:", cv_scores.mean()) # average performance

Cross Validation Accuracy: 0.9736263736263737


In [45]:
print("\nFinal Model Selected: Tuned SVM")


Final Model Selected: Tuned SVM


In [46]:
joblib.dump(best_model, "model.pkl")
joblib.dump(best_model.named_steps["scaler"], "scaler.pkl")

print("\nFinal model and scaler saved successfully")


Final model and scaler saved successfully
